In [0]:
%run ./00_config

## Ingest csv and write to delta

In [0]:
FILE_NAME = "wikipedia_movies.csv"
TABLE_NAME = "wikipedia"

In [0]:
df = spark.read.csv(f"/Volumes/hannamoazam_catalog/cookbook/source_docs/{FILE_NAME}",   
                     header=True,
                     inferSchema=True,
                     quote='"',
                     escape='"',
                     nullValue='unknown',
                     multiLine=True)

display(df)

In [0]:
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn("id", monotonically_increasing_id())
display(df)

In [0]:
from pyspark.sql.functions import col

# Replace invalid characters in column names
for col_name in df.columns:
    new_col_name = col_name.translate(str.maketrans(" ,;{}()\n\t=", "__________"))
    df = df.withColumnRenamed(col_name, new_col_name)



In [0]:
df = df.withColumnRenamed("origin/ethnicity", "origin")
display(df)

In [0]:
table_name = f"{UC_CATALOG}.{UC_SCHEMA}.{TABLE_NAME}"
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

In [0]:
spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES (delta.enableChangeDataCapture = true)")